In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import model_20250901
import simulate
from sklearn.metrics import r2_score
import importlib
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
import random


In [2]:
import model_20250901
importlib.reload(model_20250901)      # Force-reload the module
from model_20250901 import CNN_3_3_16_512, CNN_2_2_16_512, CNN_3_3_16_256 , CNN_2_2_16_256 , CNN_3_3_8_256, CNN_2_2_8_256, CNN_3_3_8_128, CNN_2_2_8_128, CNN_3_3_16_128, CNN_2_2_16_128       # Import the updated class

In [3]:
#1 
def run_trial(trial_seed):
    device = "cpu" # the device on which the model is trained, can be "cpu", "cuda" or "mps" ("mps" is only available for mac with M-series chip)
    random_seed = trial_seed
    r2 = 0.5 # true r2 of the simulated data
    n = 1000 # simulation sample size
    dim = 112 # dimensions of the simulated images
    coord, true_beta, img_data, y = simulate.simulate_data(n, r2, dim, random_seed)
    # print("First 100 Y values:")
    # print(y[:100])

    # reshape image from 1d to 2d
    img_data_0_reshaped = img_data[0].reshape(n, dim, dim)
    img_data_1_reshaped = img_data[1].reshape(n, dim, dim)

    # stack image 1 and image 2 for each observation
    stacked_img = np.concatenate([img_data_0_reshaped, img_data_1_reshaped], axis = 1)
    stacked_img = stacked_img[:, np.newaxis, :, :]

    # input_height = stacked_img.shape[2] 
    # input_width = stacked_img.shape[3] 

    # create torch tensors
    y = y.reshape(-1, 1)
    y_tensor = torch.tensor(y, dtype = torch.float32).to(device)
    stacked_img_tensor = torch.tensor(stacked_img, dtype = torch.float32).to(device)

    # set random seed
    torch.manual_seed(random_seed)
    np.random.seed(random_seed)

    # split training and testing set and pass them into torch dataloaders
    X_train, X_test, y_train, y_test = train_test_split(stacked_img_tensor, y_tensor, test_size = 0.2, random_state = random_seed)
    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size = 16, shuffle = True)
    test_loader = DataLoader(test_dataset, batch_size = 16, shuffle = False)

    # training
    cnn = model_20250901.CNN_3_3_16_512().to(device)
    criterion = torch.nn.MSELoss()
    optimizer = optim.Adam(cnn.parameters(), lr = 0.0001, weight_decay=0.0)


    num_epochs = 20
    for epoch in range(num_epochs):
        cnn.train()
        running_loss = 0.0
        
        # train for one epoch
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_hat_batch = cnn(X_batch)
            loss = criterion(y_hat_batch, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        # evaluate and print every 5 epochs
        if (epoch + 1) % 5 == 0:
            cnn.eval()
            with torch.no_grad():
                y_train_pred = cnn(X_train).cpu().numpy().flatten()
                y_train_true = y_train.cpu().numpy().flatten()
                train_r2 = np.corrcoef(y_train_true, y_train_pred)[0, 1] ** 2
                
                y_test_pred = cnn(X_test).cpu().numpy().flatten()
                y_test_true = y_test.cpu().numpy().flatten()
                test_r2 = np.corrcoef(y_test_true, y_test_pred)[0, 1] ** 2
                test_mse = np.mean((y_test_true - y_test_pred) ** 2)

                # print(f"Epoch {epoch + 1}, Loss: {running_loss / len(train_loader):.4f}, Train R²: {train_r2:.4f}")
                # print(f"Epoch {epoch + 1}, Test R²: {test_r2:.4f}")


    cnn.eval()
    with torch.no_grad():
        y_train_pred = cnn(X_train).cpu().numpy().flatten()
        y_test_pred = cnn(X_test).cpu().numpy().flatten()

    y_train_true = y_train.cpu().numpy().flatten()
    y_test_true = y_test.cpu().numpy().flatten()

    train_mse = np.mean((y_train_true - y_train_pred) ** 2)
    test_mse = np.mean((y_test_true - y_test_pred) ** 2)
    
    train_r2 = np.corrcoef(y_train_true, y_train_pred)[0, 1] ** 2
    test_r2 = np.corrcoef(y_test_true, y_test_pred)[0, 1] ** 2

    

    return train_mse, test_mse, train_r2, test_r2



def main():
    num_trials = 50
    results = []
    for i in range(num_trials):
        print(f"=== Test {i} Running ===")
        trial_seed = random.randint(1000, 9999) 
        result = run_trial(trial_seed=2025 + i)
        results.append(result)

    train_mse_list, test_mse_list, train_r2_list, test_r2_list = zip(*results)

    print(f"=== Summary over {num_trials} trials ===")
    print("Train MSE: mean = %.4f, std = %.4f" % (np.mean(train_mse_list), np.std(train_mse_list)))
    print("Test MSE:  mean = %.4f, std = %.4f" % (np.mean(test_mse_list), np.std(test_mse_list)))
    print("Train R²:  mean = %.4f, std = %.4f" % (np.mean(train_r2_list), np.std(train_r2_list)))
    print("Test R²:   mean = %.4f, std = %.4f" % (np.mean(test_r2_list), np.std(test_r2_list)))

if __name__ == "__main__":
    main()

=== Test 0 Running ===
=== Test 1 Running ===
=== Test 2 Running ===
=== Test 3 Running ===
=== Test 4 Running ===
=== Test 5 Running ===
=== Test 6 Running ===
=== Test 7 Running ===
=== Test 8 Running ===
=== Test 9 Running ===
=== Test 10 Running ===
=== Test 11 Running ===
=== Test 12 Running ===
=== Test 13 Running ===
=== Test 14 Running ===
=== Test 15 Running ===
=== Test 16 Running ===
=== Test 17 Running ===
=== Test 18 Running ===
=== Test 19 Running ===
=== Test 20 Running ===
=== Test 21 Running ===
=== Test 22 Running ===
=== Test 23 Running ===
=== Test 24 Running ===
=== Test 25 Running ===
=== Test 26 Running ===
=== Test 27 Running ===
=== Test 28 Running ===
=== Test 29 Running ===
=== Test 30 Running ===
=== Test 31 Running ===
=== Test 32 Running ===
=== Test 33 Running ===
=== Test 34 Running ===
=== Test 35 Running ===
=== Test 36 Running ===
=== Test 37 Running ===
=== Test 38 Running ===
=== Test 39 Running ===
=== Test 40 Running ===
=== Test 41 Running ===
==

In [4]:
#2

def run_trial(trial_seed):
    device = "cpu" # the device on which the model is trained, can be "cpu", "cuda" or "mps" ("mps" is only available for mac with M-series chip)
    random_seed = trial_seed
    r2 = 0.5 # true r2 of the simulated data
    n = 1000 # simulation sample size
    dim = 112 # dimensions of the simulated images
    coord, true_beta, img_data, y = simulate.simulate_data(n, r2, dim, random_seed)
    # print("First 100 Y values:")
    # print(y[:100])

    # reshape image from 1d to 2d
    img_data_0_reshaped = img_data[0].reshape(n, dim, dim)
    img_data_1_reshaped = img_data[1].reshape(n, dim, dim)

    # stack image 1 and image 2 for each observation
    stacked_img = np.concatenate([img_data_0_reshaped, img_data_1_reshaped], axis = 1)
    stacked_img = stacked_img[:, np.newaxis, :, :]

    # input_height = stacked_img.shape[2] 
    # input_width = stacked_img.shape[3] 

    # create torch tensors
    y = y.reshape(-1, 1)
    y_tensor = torch.tensor(y, dtype = torch.float32).to(device)
    stacked_img_tensor = torch.tensor(stacked_img, dtype = torch.float32).to(device)

    # set random seed
    torch.manual_seed(random_seed)
    np.random.seed(random_seed)

    # split training and testing set and pass them into torch dataloaders
    X_train, X_test, y_train, y_test = train_test_split(stacked_img_tensor, y_tensor, test_size = 0.2, random_state = random_seed)
    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size = 16, shuffle = True)
    test_loader = DataLoader(test_dataset, batch_size = 16, shuffle = False)

    # training
    cnn = model_20250901.CNN_3_3_16_256().to(device)
    criterion = torch.nn.MSELoss()
    optimizer = optim.Adam(cnn.parameters(), lr = 0.001, weight_decay=0.0)


    num_epochs = 20
    for epoch in range(num_epochs):
        cnn.train()
        running_loss = 0.0
        
        # train for one epoch
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_hat_batch = cnn(X_batch)
            loss = criterion(y_hat_batch, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        # evaluate and print every 5 epochs
        if (epoch + 1) % 5 == 0:
            cnn.eval()
            with torch.no_grad():
                y_train_pred = cnn(X_train).cpu().numpy().flatten()
                y_train_true = y_train.cpu().numpy().flatten()
                train_r2 = np.corrcoef(y_train_true, y_train_pred)[0, 1] ** 2
                
                y_test_pred = cnn(X_test).cpu().numpy().flatten()
                y_test_true = y_test.cpu().numpy().flatten()
                test_r2 = np.corrcoef(y_test_true, y_test_pred)[0, 1] ** 2
                test_mse = np.mean((y_test_true - y_test_pred) ** 2)

                # print(f"Epoch {epoch + 1}, Loss: {running_loss / len(train_loader):.4f}, Train R²: {train_r2:.4f}")
                # print(f"Epoch {epoch + 1}, Test R²: {test_r2:.4f}")


    cnn.eval()
    with torch.no_grad():
        y_train_pred = cnn(X_train).cpu().numpy().flatten()
        y_test_pred = cnn(X_test).cpu().numpy().flatten()

    y_train_true = y_train.cpu().numpy().flatten()
    y_test_true = y_test.cpu().numpy().flatten()

    train_mse = np.mean((y_train_true - y_train_pred) ** 2)
    test_mse = np.mean((y_test_true - y_test_pred) ** 2)
    
    train_r2 = np.corrcoef(y_train_true, y_train_pred)[0, 1] ** 2
    test_r2 = np.corrcoef(y_test_true, y_test_pred)[0, 1] ** 2

    

    return train_mse, test_mse, train_r2, test_r2



def main():
    num_trials = 50
    results = []
    for i in range(num_trials):
        print(f"=== Test {i} Running ===")
        trial_seed = random.randint(1000, 9999) 
        result = run_trial(trial_seed=2025 + i)
        results.append(result)

    train_mse_list, test_mse_list, train_r2_list, test_r2_list = zip(*results)

    print(f"=== Summary over {num_trials} trials ===")
    print("Train MSE: mean = %.4f, std = %.4f" % (np.mean(train_mse_list), np.std(train_mse_list)))
    print("Test MSE:  mean = %.4f, std = %.4f" % (np.mean(test_mse_list), np.std(test_mse_list)))
    print("Train R²:  mean = %.4f, std = %.4f" % (np.mean(train_r2_list), np.std(train_r2_list)))
    print("Test R²:   mean = %.4f, std = %.4f" % (np.mean(test_r2_list), np.std(test_r2_list)))

if __name__ == "__main__":
    main()

=== Test 0 Running ===
=== Test 1 Running ===
=== Test 2 Running ===
=== Test 3 Running ===
=== Test 4 Running ===
=== Test 5 Running ===
=== Test 6 Running ===
=== Test 7 Running ===
=== Test 8 Running ===
=== Test 9 Running ===
=== Test 10 Running ===
=== Test 11 Running ===
=== Test 12 Running ===
=== Test 13 Running ===
=== Test 14 Running ===
=== Test 15 Running ===
=== Test 16 Running ===
=== Test 17 Running ===
=== Test 18 Running ===
=== Test 19 Running ===
=== Test 20 Running ===
=== Test 21 Running ===
=== Test 22 Running ===
=== Test 23 Running ===
=== Test 24 Running ===
=== Test 25 Running ===
=== Test 26 Running ===
=== Test 27 Running ===
=== Test 28 Running ===
=== Test 29 Running ===
=== Test 30 Running ===
=== Test 31 Running ===
=== Test 32 Running ===
=== Test 33 Running ===
=== Test 34 Running ===
=== Test 35 Running ===
=== Test 36 Running ===
=== Test 37 Running ===
=== Test 38 Running ===
=== Test 39 Running ===
=== Test 40 Running ===
=== Test 41 Running ===
==

In [5]:
#3

def run_trial(trial_seed):
    device = "cpu" # the device on which the model is trained, can be "cpu", "cuda" or "mps" ("mps" is only available for mac with M-series chip)
    random_seed = trial_seed
    r2 = 0.5 # true r2 of the simulated data
    n = 1000 # simulation sample size
    dim = 112 # dimensions of the simulated images
    coord, true_beta, img_data, y = simulate.simulate_data(n, r2, dim, random_seed)
    # print("First 100 Y values:")
    # print(y[:100])

    # reshape image from 1d to 2d
    img_data_0_reshaped = img_data[0].reshape(n, dim, dim)
    img_data_1_reshaped = img_data[1].reshape(n, dim, dim)

    # stack image 1 and image 2 for each observation
    stacked_img = np.concatenate([img_data_0_reshaped, img_data_1_reshaped], axis = 1)
    stacked_img = stacked_img[:, np.newaxis, :, :]

    # input_height = stacked_img.shape[2] 
    # input_width = stacked_img.shape[3] 

    # create torch tensors
    y = y.reshape(-1, 1)
    y_tensor = torch.tensor(y, dtype = torch.float32).to(device)
    stacked_img_tensor = torch.tensor(stacked_img, dtype = torch.float32).to(device)

    # set random seed
    torch.manual_seed(random_seed)
    np.random.seed(random_seed)

    # split training and testing set and pass them into torch dataloaders
    X_train, X_test, y_train, y_test = train_test_split(stacked_img_tensor, y_tensor, test_size = 0.2, random_state = random_seed)
    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size = 16, shuffle = True)
    test_loader = DataLoader(test_dataset, batch_size = 16, shuffle = False)

    # training
    cnn = model_20250901.CNN_3_3_8_256().to(device)
    criterion = torch.nn.MSELoss()
    optimizer = optim.Adam(cnn.parameters(), lr = 0.001, weight_decay=0.0)


    num_epochs = 20
    for epoch in range(num_epochs):
        cnn.train()
        running_loss = 0.0
        
        # train for one epoch
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_hat_batch = cnn(X_batch)
            loss = criterion(y_hat_batch, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        # evaluate and print every 5 epochs
        if (epoch + 1) % 5 == 0:
            cnn.eval()
            with torch.no_grad():
                y_train_pred = cnn(X_train).cpu().numpy().flatten()
                y_train_true = y_train.cpu().numpy().flatten()
                train_r2 = np.corrcoef(y_train_true, y_train_pred)[0, 1] ** 2
                
                y_test_pred = cnn(X_test).cpu().numpy().flatten()
                y_test_true = y_test.cpu().numpy().flatten()
                test_r2 = np.corrcoef(y_test_true, y_test_pred)[0, 1] ** 2
                test_mse = np.mean((y_test_true - y_test_pred) ** 2)

                # print(f"Epoch {epoch + 1}, Loss: {running_loss / len(train_loader):.4f}, Train R²: {train_r2:.4f}")
                # print(f"Epoch {epoch + 1}, Test R²: {test_r2:.4f}")


    cnn.eval()
    with torch.no_grad():
        y_train_pred = cnn(X_train).cpu().numpy().flatten()
        y_test_pred = cnn(X_test).cpu().numpy().flatten()

    y_train_true = y_train.cpu().numpy().flatten()
    y_test_true = y_test.cpu().numpy().flatten()

    train_mse = np.mean((y_train_true - y_train_pred) ** 2)
    test_mse = np.mean((y_test_true - y_test_pred) ** 2)
    
    train_r2 = np.corrcoef(y_train_true, y_train_pred)[0, 1] ** 2
    test_r2 = np.corrcoef(y_test_true, y_test_pred)[0, 1] ** 2

    

    return train_mse, test_mse, train_r2, test_r2



def main():
    num_trials = 50
    results = []
    for i in range(num_trials):
        print(f"=== Test {i} Running ===")
        trial_seed = random.randint(1000, 9999) 
        result = run_trial(trial_seed=2025 + i)
        results.append(result)

    train_mse_list, test_mse_list, train_r2_list, test_r2_list = zip(*results)

    print(f"=== Summary over {num_trials} trials ===")
    print("Train MSE: mean = %.4f, std = %.4f" % (np.mean(train_mse_list), np.std(train_mse_list)))
    print("Test MSE:  mean = %.4f, std = %.4f" % (np.mean(test_mse_list), np.std(test_mse_list)))
    print("Train R²:  mean = %.4f, std = %.4f" % (np.mean(train_r2_list), np.std(train_r2_list)))
    print("Test R²:   mean = %.4f, std = %.4f" % (np.mean(test_r2_list), np.std(test_r2_list)))

if __name__ == "__main__":
    main()

=== Test 0 Running ===
=== Test 1 Running ===
=== Test 2 Running ===
=== Test 3 Running ===
=== Test 4 Running ===
=== Test 5 Running ===
=== Test 6 Running ===
=== Test 7 Running ===
=== Test 8 Running ===
=== Test 9 Running ===
=== Test 10 Running ===
=== Test 11 Running ===
=== Test 12 Running ===
=== Test 13 Running ===
=== Test 14 Running ===
=== Test 15 Running ===
=== Test 16 Running ===
=== Test 17 Running ===
=== Test 18 Running ===
=== Test 19 Running ===
=== Test 20 Running ===
=== Test 21 Running ===
=== Test 22 Running ===
=== Test 23 Running ===
=== Test 24 Running ===
=== Test 25 Running ===
=== Test 26 Running ===
=== Test 27 Running ===
=== Test 28 Running ===
=== Test 29 Running ===
=== Test 30 Running ===
=== Test 31 Running ===
=== Test 32 Running ===
=== Test 33 Running ===
=== Test 34 Running ===
=== Test 35 Running ===
=== Test 36 Running ===
=== Test 37 Running ===
=== Test 38 Running ===
=== Test 39 Running ===
=== Test 40 Running ===
=== Test 41 Running ===
==

In [6]:
#4

def run_trial(trial_seed):
    device = "cpu" # the device on which the model is trained, can be "cpu", "cuda" or "mps" ("mps" is only available for mac with M-series chip)
    random_seed = trial_seed
    r2 = 0.5 # true r2 of the simulated data
    n = 1000 # simulation sample size
    dim = 112 # dimensions of the simulated images
    coord, true_beta, img_data, y = simulate.simulate_data(n, r2, dim, random_seed)
    # print("First 100 Y values:")
    # print(y[:100])

    # reshape image from 1d to 2d
    img_data_0_reshaped = img_data[0].reshape(n, dim, dim)
    img_data_1_reshaped = img_data[1].reshape(n, dim, dim)

    # stack image 1 and image 2 for each observation
    stacked_img = np.concatenate([img_data_0_reshaped, img_data_1_reshaped], axis = 1)
    stacked_img = stacked_img[:, np.newaxis, :, :]

    # input_height = stacked_img.shape[2] 
    # input_width = stacked_img.shape[3] 

    # create torch tensors
    y = y.reshape(-1, 1)
    y_tensor = torch.tensor(y, dtype = torch.float32).to(device)
    stacked_img_tensor = torch.tensor(stacked_img, dtype = torch.float32).to(device)

    # set random seed
    torch.manual_seed(random_seed)
    np.random.seed(random_seed)

    # split training and testing set and pass them into torch dataloaders
    X_train, X_test, y_train, y_test = train_test_split(stacked_img_tensor, y_tensor, test_size = 0.2, random_state = random_seed)
    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size = 16, shuffle = True)
    test_loader = DataLoader(test_dataset, batch_size = 16, shuffle = False)

    # training
    cnn = model_20250901.CNN_2_2_8_256().to(device)
    criterion = torch.nn.MSELoss()
    optimizer = optim.Adam(cnn.parameters(), lr = 0.001, weight_decay=0.0)


    num_epochs = 20
    for epoch in range(num_epochs):
        cnn.train()
        running_loss = 0.0
        
        # train for one epoch
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_hat_batch = cnn(X_batch)
            loss = criterion(y_hat_batch, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        # evaluate and print every 5 epochs
        if (epoch + 1) % 5 == 0:
            cnn.eval()
            with torch.no_grad():
                y_train_pred = cnn(X_train).cpu().numpy().flatten()
                y_train_true = y_train.cpu().numpy().flatten()
                train_r2 = np.corrcoef(y_train_true, y_train_pred)[0, 1] ** 2
                
                y_test_pred = cnn(X_test).cpu().numpy().flatten()
                y_test_true = y_test.cpu().numpy().flatten()
                test_r2 = np.corrcoef(y_test_true, y_test_pred)[0, 1] ** 2
                test_mse = np.mean((y_test_true - y_test_pred) ** 2)

                # print(f"Epoch {epoch + 1}, Loss: {running_loss / len(train_loader):.4f}, Train R²: {train_r2:.4f}")
                # print(f"Epoch {epoch + 1}, Test R²: {test_r2:.4f}")


    cnn.eval()
    with torch.no_grad():
        y_train_pred = cnn(X_train).cpu().numpy().flatten()
        y_test_pred = cnn(X_test).cpu().numpy().flatten()

    y_train_true = y_train.cpu().numpy().flatten()
    y_test_true = y_test.cpu().numpy().flatten()

    train_mse = np.mean((y_train_true - y_train_pred) ** 2)
    test_mse = np.mean((y_test_true - y_test_pred) ** 2)
    
    train_r2 = np.corrcoef(y_train_true, y_train_pred)[0, 1] ** 2
    test_r2 = np.corrcoef(y_test_true, y_test_pred)[0, 1] ** 2

    

    return train_mse, test_mse, train_r2, test_r2



def main():
    num_trials = 50
    results = []
    for i in range(num_trials):
        print(f"=== Test {i} Running ===")
        trial_seed = random.randint(1000, 9999) 
        result = run_trial(trial_seed=2025 + i)
        results.append(result)

    train_mse_list, test_mse_list, train_r2_list, test_r2_list = zip(*results)

    print(f"=== Summary over {num_trials} trials ===")
    print("Train MSE: mean = %.4f, std = %.4f" % (np.mean(train_mse_list), np.std(train_mse_list)))
    print("Test MSE:  mean = %.4f, std = %.4f" % (np.mean(test_mse_list), np.std(test_mse_list)))
    print("Train R²:  mean = %.4f, std = %.4f" % (np.mean(train_r2_list), np.std(train_r2_list)))
    print("Test R²:   mean = %.4f, std = %.4f" % (np.mean(test_r2_list), np.std(test_r2_list)))

if __name__ == "__main__":
    main()

=== Test 0 Running ===
=== Test 1 Running ===
=== Test 2 Running ===
=== Test 3 Running ===
=== Test 4 Running ===
=== Test 5 Running ===
=== Test 6 Running ===
=== Test 7 Running ===
=== Test 8 Running ===
=== Test 9 Running ===
=== Test 10 Running ===
=== Test 11 Running ===
=== Test 12 Running ===
=== Test 13 Running ===
=== Test 14 Running ===
=== Test 15 Running ===
=== Test 16 Running ===
=== Test 17 Running ===
=== Test 18 Running ===
=== Test 19 Running ===
=== Test 20 Running ===
=== Test 21 Running ===
=== Test 22 Running ===
=== Test 23 Running ===
=== Test 24 Running ===
=== Test 25 Running ===
=== Test 26 Running ===
=== Test 27 Running ===
=== Test 28 Running ===
=== Test 29 Running ===
=== Test 30 Running ===
=== Test 31 Running ===
=== Test 32 Running ===
=== Test 33 Running ===
=== Test 34 Running ===
=== Test 35 Running ===
=== Test 36 Running ===
=== Test 37 Running ===
=== Test 38 Running ===
=== Test 39 Running ===
=== Test 40 Running ===
=== Test 41 Running ===
==

In [7]:
#5

def run_trial(trial_seed):
    device = "cpu" # the device on which the model is trained, can be "cpu", "cuda" or "mps" ("mps" is only available for mac with M-series chip)
    random_seed = trial_seed
    r2 = 0.5 # true r2 of the simulated data
    n = 1000 # simulation sample size
    dim = 112 # dimensions of the simulated images
    coord, true_beta, img_data, y = simulate.simulate_data(n, r2, dim, random_seed)
    # print("First 100 Y values:")
    # print(y[:100])

    # reshape image from 1d to 2d
    img_data_0_reshaped = img_data[0].reshape(n, dim, dim)
    img_data_1_reshaped = img_data[1].reshape(n, dim, dim)

    # stack image 1 and image 2 for each observation
    stacked_img = np.concatenate([img_data_0_reshaped, img_data_1_reshaped], axis = 1)
    stacked_img = stacked_img[:, np.newaxis, :, :]

    # input_height = stacked_img.shape[2] 
    # input_width = stacked_img.shape[3] 

    # create torch tensors
    y = y.reshape(-1, 1)
    y_tensor = torch.tensor(y, dtype = torch.float32).to(device)
    stacked_img_tensor = torch.tensor(stacked_img, dtype = torch.float32).to(device)

    # set random seed
    torch.manual_seed(random_seed)
    np.random.seed(random_seed)

    # split training and testing set and pass them into torch dataloaders
    X_train, X_test, y_train, y_test = train_test_split(stacked_img_tensor, y_tensor, test_size = 0.2, random_state = random_seed)
    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size = 16, shuffle = True)
    test_loader = DataLoader(test_dataset, batch_size = 16, shuffle = False)

    # training
    cnn = model_20250901.CNN_3_3_16_128().to(device)
    criterion = torch.nn.MSELoss()
    optimizer = optim.Adam(cnn.parameters(), lr = 0.001, weight_decay=0.0)


    num_epochs = 20
    for epoch in range(num_epochs):
        cnn.train()
        running_loss = 0.0
        
        # train for one epoch
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_hat_batch = cnn(X_batch)
            loss = criterion(y_hat_batch, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        # evaluate and print every 5 epochs
        if (epoch + 1) % 5 == 0:
            cnn.eval()
            with torch.no_grad():
                y_train_pred = cnn(X_train).cpu().numpy().flatten()
                y_train_true = y_train.cpu().numpy().flatten()
                train_r2 = np.corrcoef(y_train_true, y_train_pred)[0, 1] ** 2
                
                y_test_pred = cnn(X_test).cpu().numpy().flatten()
                y_test_true = y_test.cpu().numpy().flatten()
                test_r2 = np.corrcoef(y_test_true, y_test_pred)[0, 1] ** 2
                test_mse = np.mean((y_test_true - y_test_pred) ** 2)

                # print(f"Epoch {epoch + 1}, Loss: {running_loss / len(train_loader):.4f}, Train R²: {train_r2:.4f}")
                # print(f"Epoch {epoch + 1}, Test R²: {test_r2:.4f}")


    cnn.eval()
    with torch.no_grad():
        y_train_pred = cnn(X_train).cpu().numpy().flatten()
        y_test_pred = cnn(X_test).cpu().numpy().flatten()

    y_train_true = y_train.cpu().numpy().flatten()
    y_test_true = y_test.cpu().numpy().flatten()

    train_mse = np.mean((y_train_true - y_train_pred) ** 2)
    test_mse = np.mean((y_test_true - y_test_pred) ** 2)
    
    train_r2 = np.corrcoef(y_train_true, y_train_pred)[0, 1] ** 2
    test_r2 = np.corrcoef(y_test_true, y_test_pred)[0, 1] ** 2

    

    return train_mse, test_mse, train_r2, test_r2



def main():
    num_trials = 50
    results = []
    for i in range(num_trials):
        print(f"=== Test {i} Running ===")
        trial_seed = random.randint(1000, 9999) 
        result = run_trial(trial_seed=2025 + i)
        results.append(result)

    train_mse_list, test_mse_list, train_r2_list, test_r2_list = zip(*results)

    print(f"=== Summary over {num_trials} trials ===")
    print("Train MSE: mean = %.4f, std = %.4f" % (np.mean(train_mse_list), np.std(train_mse_list)))
    print("Test MSE:  mean = %.4f, std = %.4f" % (np.mean(test_mse_list), np.std(test_mse_list)))
    print("Train R²:  mean = %.4f, std = %.4f" % (np.mean(train_r2_list), np.std(train_r2_list)))
    print("Test R²:   mean = %.4f, std = %.4f" % (np.mean(test_r2_list), np.std(test_r2_list)))

if __name__ == "__main__":
    main()

=== Test 0 Running ===
=== Test 1 Running ===
=== Test 2 Running ===
=== Test 3 Running ===
=== Test 4 Running ===
=== Test 5 Running ===
=== Test 6 Running ===
=== Test 7 Running ===
=== Test 8 Running ===
=== Test 9 Running ===
=== Test 10 Running ===
=== Test 11 Running ===
=== Test 12 Running ===
=== Test 13 Running ===
=== Test 14 Running ===
=== Test 15 Running ===
=== Test 16 Running ===
=== Test 17 Running ===
=== Test 18 Running ===
=== Test 19 Running ===
=== Test 20 Running ===
=== Test 21 Running ===
=== Test 22 Running ===
=== Test 23 Running ===
=== Test 24 Running ===
=== Test 25 Running ===
=== Test 26 Running ===
=== Test 27 Running ===
=== Test 28 Running ===
=== Test 29 Running ===
=== Test 30 Running ===
=== Test 31 Running ===
=== Test 32 Running ===
=== Test 33 Running ===
=== Test 34 Running ===
=== Test 35 Running ===
=== Test 36 Running ===
=== Test 37 Running ===
=== Test 38 Running ===
=== Test 39 Running ===
=== Test 40 Running ===
=== Test 41 Running ===
==